# SER - CNN

In [1]:
import matplotlib.pyplot as plt
import librosa
import numpy as np
import os, sys
module_path = os.path.abspath(os.path.join('..', '..')) 
sys.path.insert(0, module_path)
from src.data_loading import load_emodb, load_ravdess, load_tess, load_crema_d, load_meld, filter_emotions, load_iemocap
from src.evaluation import plot_confusion_matrix, plot_training_history

emodb = filter_emotions(load_emodb())
ravdess = filter_emotions(load_ravdess())
tess = filter_emotions(load_tess())
crema = filter_emotions(load_crema_d())
iemocap = filter_emotions(load_iemocap())


/Users/krazmic/Documents/GitHub/Repos/EmoReA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/krazmic/.cache/kagglehub/datasets/piyushagni5/berlin-database-of-emotional-speech-emodb/versions/1
Path to dataset files: /Users/krazmic/.cache/kagglehub/datasets/uwrfkaggler/ravdess-emotional-speech-audio/versions/1
['Actor_16', 'Actor_11', 'Actor_18', 'Actor_20', 'Actor_21', 'Actor_19', 'Actor_10', 'Actor_17', 'Actor_04', 'Actor_03', 'Actor_02', 'Actor_05', 'audio_speech_actors_01-24', 'Actor_12', 'Actor_15', 'Actor_23', 'Actor_24', 'Actor_22', 'Actor_14', 'Actor_13', 'Actor_09', 'Actor_07', 'Actor_06', 'Actor_01', 'Actor_08']
Path to dataset files: /Users/krazmic/.cache/kagglehub/datasets/ejlok1/toronto-emotional-speech-set-tess/versions/1
['TESS']
['YAF_disgust', 'OAF_Pleasant_surprise', 'OAF_happy', 'YAF_sad', 'TESS Toronto emotional speech set data', 'YAF_happy', 'YAF_neutral', 'OAF_Fear', 'OAF_angry', 'YAF_pleasant_surprised', 'YAF_fear', 'OAF_neutral', 'OAF_disgust', 'YAF_angry', 'OAF_Sad']
Path to dataset files: /Users/krazmic/.cache/kagglehub/dat

In [3]:
y, _ = librosa.load(emodb.iloc[0]['filename'], sr=None)
S = librosa.feature.melspectrogram(y=y, sr=_, n_mels=128)
S.shape

(128, 65)

In [7]:
# 
# import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, BatchNormalization, Input
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
from tqdm import tqdm
tqdm.pandas()

# -----------------------------
# Dataset preparation
# -----------------------------
def prepare_dataset(df, sr=16000, n_mels=128, resize_shape=(128, 256)):
    """
    df: DataFrame with 'filename' and 'label' columns
    sr: sampling rate
    n_mels: number of STFT bins (frequency axis)
    resize_shape: (height, width) of STFT images for CNN
    """
    # Encode labels
    le = LabelEncoder()
    df['label_enc'] = le.fit_transform(df['label'])
    labels = df['label_enc'].values
    n_classes = len(le.classes_)

    stft_list = []
    for fname in tqdm(df['filename']):
        y, _ = librosa.load(fname, sr=sr)
        #S = np.abs(librosa.stft(y))
        #S_db = np.log1p(S)  # log-scale
        # Resize to fixed shape
        #S_resized = np.resize(S_db, resize_shape)
        # compute spectrogram
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)#len(y)//512)#128)
        S_dB = librosa.power_to_db(S, ref=np.max)

        S_resized = np.resize(S_dB, (128, 256))
        # Add channel dimension
        S_resized = S_resized[..., np.newaxis]
        stft_list.append(S_resized)

    X = np.array(stft_list, dtype=np.float32)
    le = LabelEncoder()
    y = le.fit_transform(labels)
    #y = tf.keras.utils.to_categorical(labels, num_classes=n_classes)

    return X, y, n_classes, le

# Prepare datasets
df = pd.concat([emodb])
X = df['filename']
y = df['label']
train_df, val_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(val_df, test_size=0.5, random_state=42, stratify=val_df['label'])

X_train, y_train, n_classes, label_encoder = prepare_dataset(train_df)
X_valid, y_valid, _, _ = prepare_dataset(val_df)
X_test, y_test, _, _ = prepare_dataset(test_df)


100%|██████████| 69/69 [00:00<00:00, 195.89it/s]


In [ ]:


# -----------------------------
# CNN model definition
# -----------------------------
def build_light_cnn(input_shape=(48, 48, 1), num_classes=7):
    """
    Lightweight CNN for emotion recognition from frames.
    """
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(256, (3,3), activation='relu', padding='same',),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Flatten(),
        Dropout(0.7),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

class AudioCNN(tf.keras.Model):
    def __init__(self, input_shape=(128, 256, 1), n_classes=6): # input_shape=(128, 256, 1)
        super(AudioCNN, self).__init__()
        self.conv1 = tf.keras.layers.Conv2D(256, (3,3), activation='relu', padding='same', input_shape=input_shape)
        self.pool1 = tf.keras.layers.MaxPooling2D((2,2))
        self.conv2 = tf.keras.layers.Conv2D(128, (3,3), activation='relu', padding='same')
        self.pool2 = tf.keras.layers.MaxPooling2D((2,2))
        self.conv3 = tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same')
        self.pool3 = tf.keras.layers.MaxPooling2D((2,2))
        self.flatten = tf.keras.layers.Flatten()
        self.dropout = tf.keras.layers.Dropout(0.3)
        self.fc = tf.keras.layers.Dense(n_classes, activation='softmax')

    def call(self, x):
        x = self.conv1(x)
        x = self.dropout(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.dropout(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x = self.pool3(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x



# Create model
cnn_model = AudioCNN(input_shape=X_train.shape[1:], n_classes=n_classes)
epochs = 100
# Create an Adam optimizer with a decay rate
initial_learning_rate = 0.001
decay_rate = initial_learning_rate / epochs

# To use an ExponentialDecay schedule
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate,
    decay_steps=X_train.shape[0] / 16, # number of steps per epoch
    decay_rate=0.96,
    staircase=True)

# Compile the model with the customized optimizer
cnn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Callbacks for better training control
callbacks = [
    # Save only the best model (based on validation accuracy)
    ModelCheckpoint(
    "best_cnn_model",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
    ),
    # Stop early if no improvement
    EarlyStopping(
        monitor="val_accuracy",
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate when validation stops improving
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]
# Train
cnn_model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
              batch_size=16, epochs=100, verbose=1, callbacks=callbacks)

# Predict on a single file
def predict_file(fname):
    y, _ = librosa.load(fname, sr=16000)
    S = np.abs(librosa.stft(y))
    S_db = np.log1p(S)
    S_resized = np.resize(S_db, (128,256))[..., np.newaxis]
    X = np.expand_dims(S_resized, axis=0)
    pred = cnn_model.predict(X)
    label_idx = np.argmax(pred, axis=1)[0]
    return label_idx

# Example:
# label_pred = predict_file('audio_example.wav')
# print("Predicted label index:", label_pred)


Epoch 1/100


: 

In [23]:
y_pred

array([0, 4, 4, 4, 4, 4, 0, 0, 0, 5, 4, 0, 4, 0, 3, 4, 2, 0, 0, 4, 0, 5,
       0, 5, 2, 4, 0, 5, 2, 0, 4, 0, 4, 3, 4, 0, 0, 4, 4, 0, 0, 4, 0, 4,
       0, 4, 4, 0, 0, 4, 2, 5, 4, 4, 3, 2, 4, 5, 4, 2, 5, 5, 4, 0, 0, 0,
       0, 4, 4])

In [24]:
from src.evaluation import plot_confusion_matrix, plot_training_history
from sklearn.metrics import classification_report

y_pred_probs = cnn_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

#y_pred = label_encoder.inverse_transform(y_pred)
#y_test = label_encoder.inverse_transform(y_test)

print(classification_report(y_test, y_pred))
plot_confusion_matrix(y_test, y_pred)

3/3 [==============================] - 0s 205ms/step


ValueError: Classification metrics can't handle a mix of multilabel-indicator and multiclass targets

In [31]:
# Prepare datasets
X_train, y_train, n_classes = prepare_dataset(tess)
X_valid, y_valid, _ = prepare_dataset(ravdess)

# Create model
cnn_model = AudioCNN(input_shape=X_train.shape[1:], n_classes=n_classes)
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train
cnn_model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
              batch_size=16, epochs=30)

Epoch 1/30
175/175 [==============================] - 36s 198ms/step - loss: 0.8097 - accuracy: 0.7146 - val_loss: 3.9944 - val_accuracy: 0.1835
Epoch 2/30
175/175 [==============================] - 34s 192ms/step - loss: 0.2215 - accuracy: 0.9293 - val_loss: 4.3068 - val_accuracy: 0.2011
Epoch 3/30
175/175 [==============================] - 33s 188ms/step - loss: 0.0957 - accuracy: 0.9693 - val_loss: 4.3770 - val_accuracy: 0.2171
Epoch 4/30
175/175 [==============================] - 33s 191ms/step - loss: 0.0467 - accuracy: 0.9846 - val_loss: 5.3707 - val_accuracy: 0.2171
Epoch 5/30
175/175 [==============================] - 33s 188ms/step - loss: 0.0231 - accuracy: 0.9950 - val_loss: 7.7945 - val_accuracy: 0.2300
Epoch 6/30
175/175 [==============================] - 33s 189ms/step - loss: 0.0425 - accuracy: 0.9857 - val_loss: 6.9457 - val_accuracy: 0.2139
Epoch 7/30
175/175 [==============================] - 33s 189ms/step - loss: 0.0307 - accuracy: 0.9882 - val_loss: 7.2597 - val_ac

KeyboardInterrupt: 